# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lthendral10/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
!git clone https://github.com/lthendral10/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [9]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [10]:
# Proxy target (same as Week 5 if you used this approach)
df["target"] = (df["trend_direction"] == "down").astype(int)

features = [
    "search_volume",
    "ctr",
    "avg_position",
    "days_since_last_update",
    "content_age_days",
    "engagement_rate",
    "scroll_rate"
]

X = df[features].fillna(0)
y = df["target"]

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Finding 1

The paper reports that certain search signals are associated with improved content visibility.

Methodology Question:
How was the target defined? Were the labels based on observed outcomes or a predefined rule?

---

## Finding 2

The paper shows that combining multiple search signals improves ranking quality.

Methodology Question:
Was the validation performed using grouped or time-aware data splits to avoid optimistic results?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Honest Split

The Week 5 model was evaluated using a grouped validation strategy based on client_id. This reduces the chance that information from the same client appears in both training and testing data.

In [13]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (23837, 7)
Test shape: (6163, 7)


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Leakage Audit

The feature set was reviewed to identify possible leakage.

Identifier columns such as content_id and client_id were excluded from model features.

No future-window variables were intentionally used as predictors.

The selected features are available before prediction and are intended for decision support.

In [15]:
used_features = [
    "search_volume",
    "ctr",
    "avg_position",
    "days_since_last_update",
    "content_age_days",
    "engagement_rate",
    "scroll_rate"
]

excluded_features = [
    "content_id",
    "client_id",
    "provider_used",
    "model_used"
]

print("Used Features")
print(used_features)

print("\nExcluded Features")
print(excluded_features)

Used Features
['search_volume', 'ctr', 'avg_position', 'days_since_last_update', 'content_age_days', 'engagement_rate', 'scroll_rate']

Excluded Features
['content_id', 'client_id', 'provider_used', 'model_used']


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Claim Rewrite

### Original Claim
The model accurately identifies which content pages should be updated.

### Rewritten Claim
The model identifies content pages that may benefit from review based on observed SEO and engagement signals. These results are intended to support decision-making rather than provide guaranteed recommendations.

---

### Original Claim
Updating the recommended pages will improve search performance.

### Rewritten Claim
The recommended pages show characteristics associated with declining or changing performance. Updating these pages may be useful, but the effect should be validated through further analysis and monitoring.

---

### Original Claim
The Random Forest model is better than the baseline.

### Rewritten Claim
On the evaluated dataset and validation split, the Random Forest model achieved higher measured performance than the Week 4 baseline. This observation is specific to this dataset and should not be interpreted as a universal result.

---

### Summary

Throughout this project, I used careful claim language such as **observed**, **measured**, **associated**, **directional**, and **decision-support**. I avoided words such as **proves**, **guarantees**, **always**, or **causes** because the model provides evidence for decision support rather than certainty.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.